In [1]:
import os
from pyspark.sql import SparkSession

# 1. AWS 자격증명 프로필 설정 (터미널에서 -e AWS_PROFILE=metacode 한 것과 동일한 효과)
os.environ["AWS_PROFILE"] = "metacode"

# 2. Spark Session 생성 및 Iceberg/Glue 환경 세팅
spark = (
    SparkSession.builder.appName("Iceberg-Jupyter")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://metacode-iceberg-0426/warehouse")
    .getOrCreate()
)

print("Spark Iceberg 환경 세팅 완료!")

26/05/07 19:51:04 WARN Utils: Your hostname, kkrs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.219.100 instead (on interface en0)
26/05/07 19:51:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 19:51:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/07 19:51:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Iceberg 환경 세팅 완료!


26/05/07 19:51:05 WARN SparkSession: Cannot use org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions to configure session extensions.
java.lang.ClassNotFoundException: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:469)
	at org.apache.spark.util.SparkClassUtils.classForName(SparkClassUtils.scala:41)
	at org.apache.spark.util.SparkClassUtils.classForName$(SparkClassUtils.scala:36)
	at org.apache.spark.util.Utils$.classForName(Utils.scala:94)
	at org.apache.spark.sql.SparkSession$.$anonfun$applyExtensions$2(SparkSession.scala:1367)
	at org.apache.spark.sql.SparkSession$.$anonfun$applyExtensions$2$adapted(SparkSession.scala:1365)
	at scala.collection.

In [3]:
import time

# 1. 파일 통계 확인 (Before)
print("=== Compaction 전 파일 통계 ===")
spark.sql("""
    SELECT 
        COUNT(*) AS total_files,
        AVG(file_size_in_bytes) AS avg_bytes,
        MIN(file_size_in_bytes) AS min_bytes,
        MAX(file_size_in_bytes) AS max_bytes,
        SUM(file_size_in_bytes) AS total_bytes
    FROM glue_catalog.ad_lakehouse.raw_events.files
""").show()

# 2. 쿼리 성능 측정 (Before)
print("=== Compaction 전 쿼리 성능 ===")
start = time.time()
spark.sql("""
    SELECT campaign, COUNT(*), SUM(click) 
    FROM glue_catalog.ad_lakehouse.raw_events 
    WHERE event_timestamp > current_date() - INTERVAL 1 DAY 
    GROUP BY campaign
""").collect()
print(f"Query time (before compaction): {time.time() - start:.2f}s")

# 3. EXPLAIN으로 스캔 파일 수 확인 (Before)
print("=== Compaction 전 EXPLAIN ===")
spark.sql("""
    EXPLAIN FORMATTED 
    SELECT campaign, COUNT(*) 
    FROM glue_catalog.ad_lakehouse.raw_events 
    WHERE campaign = '12345'
""").show(truncate=False)

=== Compaction 전 파일 통계 ===


Py4JJavaError: An error occurred while calling o36.sql.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'glue_catalog': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:53)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:53)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndIdentifier$.unapply(LookupCatalog.scala:122)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveRelations$.$anonfun$resolveRelation$1(Analyzer.scala:1263)
	at scala.Option.orElse(Option.scala:447)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveRelation